# Notebook 6A: Decision Trees for Classifying High Asthma Burden

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

In the last few notebooks, you built supervised learning models with the CalEnviroScreen dataset. In Notebook 4B, you used **linear regression** to predict a numerical asthma burden value. In Notebooks 5A and 5B, you changed the question into a **classification** problem by predicting whether a census tract had `High Asthma` or lower asthma burden.

Today, you will try a new type of classification model: a **decision tree**.

A decision tree is useful because it makes predictions through a sequence of readable yes/no decisions. This can be helpful in human decision-making settings, like triaging patients, flagging census tracts that may need public-health attention, or deciding which cases need a closer human review. A model should not replace expert judgment, community knowledge, or public-health decision-making, but an interpretable model can help people see what patterns the model is using.

By the end of this notebook, you will be able to:

- explain why decision trees are nonlinear models
- fit a decision tree classifier called `tree_model`
- visualize a decision tree and read the information printed inside it
- evaluate a decision tree using accuracy, a confusion matrix, precision, recall, and F1
- compare simple and more flexible trees using training and test accuracy
- describe underfitting and overfitting in practical terms

You will practice again many steps you saw in Notebook 5A and 5B, including binning `Asthma` into `High Asthma`, using `stratify=y` during the train/test split, imputing missing values, and evaluating classification predictions.

## Notebook 6A roadmap

This notebook has five main parts.

**Part 1: Prepare the data**

1. Load the CalEnviroScreen data
2. Remove rows with missing `Asthma`
3. Create the binary `High Asthma` target
4. Choose the same model features from Week 5
5. Split and impute the data

**Part 2: Understand decision trees as nonlinear models**

6. Contrast decision trees with the models you have already seen
7. Fit and visualize a one-split decision tree
8. Practice reading the printed information inside a tree

**Part 3: Fit and evaluate the main decision tree**

9. Fit a depth-3 decision tree called `tree_model`
10. Visualize the tree
11. Evaluate the tree with classification metrics

**Part 4: Compare tree depth, underfitting, and overfitting**

12. Compare depth 1, depth 3, and depth 10 trees
13. Use training and test accuracy to reason about model complexity

**Part 5: Interpret the model carefully**

14. Connect the model to public-health decision-making
15. Review what the tree can and cannot tell us

# Part 1: Prepare the data

The first part of today’s notebook should feel familiar. We will rebuild the same classification setup from Notebook 5B so that the main change today is the model type.

That practice is a useful machine learning habit. When you want to understand what changes when you try a new model, it helps to keep the target, feature set, and train/test split logic as consistent as possible.

## Step 0: Import the packages we need

Most of these packages are familiar from earlier notebooks. The new imports are `DecisionTreeClassifier` and `plot_tree`.

- `DecisionTreeClassifier` lets us build a decision tree for classification.
- `plot_tree` lets us draw the fitted tree so we can inspect its decisions.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# our preprocessing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# our metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier # new today!
from sklearn.tree import plot_tree # new today!

## Step 1: Load the CalEnviroScreen data

We will load the same public CalEnviroScreen dataset from Google Drive. Use `Census Tract` as the index column, just like you did in the previous CalEnviroScreen notebooks.

The loading code is provided so we can focus today on the decision tree model.

In [ ]:
# Instructor-provided data loading code
file_id = "1X4-6X3VKhR2jRHppI3XuVV79nhHyg8Xb"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

ces = pd.read_csv(url, index_col="Census Tract")

display(ces.head())

## Step 2: Prepare rows with known `Asthma` values

For supervised learning, every row used for training needs a known target value. Today, our target will be created from the `Asthma` column, so we need to remove rows where `Asthma` is missing.

We are not dropping unused columns here. We are only removing rows that do not have the target information needed for this classification problem.

In [ ]:
print("Original shape:", ces.shape)

# Drop rows where the Asthma value is missing
ces_sup_clean = ces.dropna(subset=["Asthma"]).copy()

print("Shape after dropping rows with missing Asthma:", ces_sup_clean.shape)

## Step 3: Create the `High Asthma` classification target

In Notebook 5B, you turned the numerical `Asthma` column into a binary classification target called `High Asthma`.

We will use the same strategy today:

- calculate the median asthma burden value
- label tracts at or above the median as class `1`
- label tracts below the median as class `0`

Class `1` means predicted or actual **high asthma burden**. Class `0` means predicted or actual **lower asthma burden**.

In [ ]:
# Find the median Asthma value
asthma_cutoff = ces_sup_clean["Asthma"].median()

# Create the binary classification target
ces_sup_clean["High Asthma"] = (ces_sup_clean["Asthma"] >= asthma_cutoff).astype(int)

print("Asthma cutoff:", asthma_cutoff)
display(ces_sup_clean[["Asthma", "High Asthma"]].head())

print("\nClass counts:")
print(ces_sup_clean["High Asthma"].value_counts())

print("\nClass proportions:")
print(ces_sup_clean["High Asthma"].value_counts(normalize=True))

## Step 4: Choose the same feature set from Week 5

We will use the same feature set from Notebooks 5A and 5B. This feature set includes environmental burden variables, socioeconomic context variables, and age-structure variables.

These are tract-level model inputs. They should not be read as simple individual-level causes of asthma. Many of these variables reflect broader structural conditions, like housing policy, disinvestment, labor conditions, access to resources, and environmental decision-making.

Some columns remain in the dataframe but are not used as model features today. Race and ethnicity percentage columns are not part of this 6A fitted model because today’s goal is to understand how a single decision tree works using the same features as before. In Notebook 6B, you will return to feature choice more directly and compare models that use different sets of context variables.

In [ ]:
# Instructor-provided feature set for today's decision tree model
model_features = [
    "Ozone",
    "PM2.5",
    "Diesel PM",
    "Drinking Water",
    "Pesticides",
    "Traffic",
    "Cleanup Sites",
    "Education",
    "Housing Burden",
    "Linguistic Isolation",
    "Poverty",
    "Unemployment",
    "Children < 10 years (%)",
    "Elderly > 64 years (%)"
]

print("Number of model features:", len(model_features))
print(model_features)

## Step 5: Create `X` and `y`

Now we separate the data by column role.

- `X` contains the selected model features
- `y` contains the classification target, `High Asthma`

This is the same structure you used in earlier supervised learning notebooks.

### <font color=0D9488>**Question 1:**</font> Create the `X` feature dataset using `model_features` and create the `y` target labels using `High Asthma`. Then print their shapes and display the first 5 rows of `X`.

In [ ]:
# Your solution here!

## Step 6: Split the rows into training and test sets

Use `train_test_split()` again. Because this is classification, we will use `stratify=y`, just like in Notebook 5B.

`stratify=y` asks sklearn to keep the class proportions similar in the training set and the test set. This dataset is already close to balanced because we used the median cutoff, but stratifying is still good practice for classification. It becomes especially important when one class is much rarer than the other.

### <font color=0D9488>**Question 2:**</font> Split `X` and `y` into training and test sets using 20% test data, `random_state=42`, and `stratify=y`. Then print the shapes of the four resulting objects.

In [ ]:
# Your solution here!

## Step 7: Impute missing feature values

Decision trees do not need feature scaling, but the basic sklearn decision tree still needs complete numerical input data. That means we still need to handle missing values before fitting the model.

We will use the same median imputation workflow from earlier notebooks:

1. fit the imputer on the training features only
2. transform both the training features and the test features
3. convert the imputed arrays back into dataframes with readable column names

Fitting the imputer only on the training data helps keep information from the test set out of the training process.

### <font color=0D9488>**Question 3:**</font> Use `SimpleImputer` to impute missing values in `X_train` and `X_test` with the median from `X_train`. Convert the arrays to final dataframes named `X_train_imputed` and `X_test_imputed`, then check that each dataframe has 0 missing values.

In [ ]:
# Your solution here!

## Why we are not scaling today

In weeks 4 and 5, you scaled before applying linear models. Scaling helped because linear models use coefficients, and coefficients are easier to compare when features are on similar scales.

Decision trees work totally differently! A tree asks threshold questions one feature at a time, like “Is `Traffic` less than or equal to this value?” Because the model is asking split questions, it does not need all features to be on the same scale.

So today’s preprocessing is simpler:

- impute missing values
- do not scale

# Part 2: Understand decision trees as nonlinear models

Earlier in SCIP, you used **k-means** and **hierarchical clustering** to group rows based on similarity. Those were unsupervised methods because there was no target variable.

Then in weeks 4 and 5, you moved into supervised learning:

- linear regression predicted a numerical target using linear relationships
- logistic regression predicted a class, but it was still a linear model

Different models are useful for different reasons. Linear and logistic regression are often good starting points because they are relatively simple, fast, and easier to interpret. They are especially useful when we want a clear baseline model or when we want to understand the general direction of relationships between features and a target.

Tree-based models are useful when we think the pattern may be more complicated. For example, asthma burden might not increase in one smooth, steady way as pollution or poverty changes. Some combinations of conditions may matter more than any one variable alone. A tree can learn rule-like patterns, like one rule for tracts with higher diesel pollution and another rule for tracts with lower diesel pollution.

A **linear model** learns a relatively simple pattern. In a rough visual sense, it tries to separate or predict using straight-line structure.

A **nonlinear model** can learn patterns that are more flexible than one straight-line trend.

A decision tree is a nonlinear model because it splits the data into smaller groups and can use different decision rules in different branches of the tree.

This does not mean tree-based models are always better! A flexible model can sometimes fit the training data very closely without doing better on new data. In machine learning, we often try more than one reasonable model and compare how well each model performs on data it did not train on.

## Step 8: What is a decision tree?

A decision tree works like a flowchart. It asks a sequence of yes/no questions about the features. Part of what is so great about tree-based models is that they map well onto human thinking.

For example, a tree might ask:

1. Is `Diesel PM` less than or equal to a certain value?
2. If yes, is `Poverty` less than or equal to a certain value?
3. If no, is `Traffic` greater than a certain value?

The answers from each datapoint sends the row down a different branch. At the end, the tree reaches a final prediction.

Tree-based models can be useful for human decision-making because the rules are more visible than many other models. For example, a public-health team might use a tree-based model to identify tracts that need more detailed review. The model would not decide policy on its own, but it could help people inspect patterns and ask better follow-up questions.

## Step 9: Fit a one-split decision tree

We will start with the simplest possible decision tree: a tree with only one split.

This is sometimes called a **decision stump**. It can ask only one yes/no question before making a prediction.

Before running the code, think about what might happen:

- If the model can ask only one question, will it probably be very flexible or very limited?
- Do you expect a one-split tree to capture all of the patterns in this dataset?

In [ ]:
# Instructor-provided first decision tree example
stump_model = DecisionTreeClassifier(max_depth=1, random_state=42)

# random_state helps make the tree reproducible if sklearn has to break ties or make random choices
# We will talk about the max_depth parameter below

# Fit the model on the imputed training data
stump_model.fit(X_train_imputed, y_train)

## Step 10: Plot the one-split tree

The code below draws the tree.

Because this is your first tree plot, the plotting code is provided. You do not need to memorize every plotting option. Focus on reading the output.

## How to read the tree plot

The next code cell will draw our first decision tree. This first tree is very small because it only gets one split. A one-split tree is sometimes called a **decision stump**.

A decision tree plot looks like a flowchart. Each box is a **node**. Some nodes ask a yes/no question, called a **split**. The paths coming out of a split are **branches**. An ending node is a **leaf**, where the tree makes a prediction. **Depth** describes how many layers of decisions the tree is allowed to make.

The top node is the **root node**, where every training row starts. To choose the root split, the tree tries many possible yes/no questions across the features in `model_features`, then picks the question that best separates the training rows into groups with different class patterns.

Each node usually shows:

- the **split rule**, which is the yes/no question
- **gini**, a class-mixing score
- **samples**, the number of training rows that reached that node
- **value**, the number of rows from each class, shown here as `[lower asthma rows, high asthma rows]`
- **class**, the class the tree would predict at that node

The colors also help you read the tree. In this plot, orange nodes lean toward `Lower Asthma`, and blue nodes lean toward `High Asthma`. Darker color means the node is more strongly dominated by one class.

You do not need to calculate gini by hand. For today, think of gini as a mixing score. A node with a more even mix of lower-asthma and high-asthma tracts has a higher gini. A node mostly containing one class has a lower gini. The tree chooses splits that reduce this class mixing.

*Note: the term **Gini** comes from Corrado Gini, who unfortunately was a fascist. We still teach it because the term remains common in machine learning software and documentation, so I want you to know what it means when people talk about it.*

Now run the next cell and look for the split rule, gini, samples, value, class, and color in each node.

In [ ]:
# Instructor-provided tree plot
plt.figure(figsize=(10, 5))

plot_tree(                      # This is how we plot our tree!
    stump_model,
    feature_names=model_features,
    class_names=["Lower Asthma", "High Asthma"],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("One-Split Decision Tree for High Asthma Classification")
plt.show()

## Reading this one-split tree

In this one-split tree, the root node asks:

`Education <= 9.917`

The left branch is labeled `True`, so rows go left if `Education` is less than or equal to `9.917`. The right branch is labeled `False`, so rows go right if `Education` is greater than `9.917`.

At the top node, all training rows are still together:

- `samples = 7258` means 7,258 training rows reached this node.
- `value = [3629, 3629]` means the training data at the top is evenly split between 3,629 lower-asthma rows and 3,629 high-asthma rows.
- `gini = 0.5` means this node is very mixed between the two classes.
- `class = Lower Asthma` is the class the tree would predict at this node. When the classes are tied, sklearn still has to choose one class.

After the split, the rows are separated into two groups.

On the left side, where `Education <= 9.917` is true:

- `samples = 3261`
- `value = [2485, 776]`
- most rows in this group are lower-asthma rows
- the tree predicts `Lower Asthma`

On the right side, where `Education <= 9.917` is false:

- `samples = 3997`
- `value = [1144, 2853]`
- most rows in this group are high-asthma rows
- the tree predicts `High Asthma`

This split does not mean education causes asthma burden. It means that, in this training dataset, this one split on the `Education` feature separated the rows into two groups with different asthma-burden patterns.

This first tree only has one split, so both bottom nodes are leaves. In a deeper tree, each group can be split again. The tree repeats the same basic process inside each branch: it looks only at the training rows that reached that node, tries many possible split rules, and chooses the next rule that best reduces class mixing for that smaller group.

When the model predicts a new row, the row starts at the top of the tree and follows the same yes/no rules until it reaches a leaf. The class shown in that leaf is the model’s prediction for that row.

# Part 3: Fit and evaluate the main decision tree

A one-split tree is easy to read, but it is probably too simple for this dataset because it can only ask one question.

Next, we will fit a tree with `max_depth=3`. The `max_depth` setting controls how many layers of decisions the tree is allowed to make. A tree with a larger `max_depth` can ask more questions and learn more detailed patterns, but it can also become harder to interpret. If a tree grows too deep, it may fit small details in the training data instead of learning patterns that generalize well to new data.

`max_depth` is an example of a machine learning **hyperparameter**. A hyperparameter is a model setting that we choose before fitting or training the model. This is different from values the model learns from the training data during fitting, like which feature and threshold to use for each split.

In real machine learning projects, data scientists often try several hyperparameter values and compare the results. This process is called **hyperparameter tuning**. Tuning means changing model settings to look for a model that performs well on new data, not just on the training data.

One important caution: in a full machine learning project, we would not keep changing hyperparameters after checking performance on the test set. The test set is supposed to act like new, unseen data. If we test the model on the test set, make tweaks, and then test again, the test set starts to influence our modeling choices. It stops being a fair final check of how the model handles new data.

A common solution, which is outside the scope of this course, is to separate some of the training data into a new dataset called a **validation set** to compare model settings. The test set is then saved for the final evaluation. We will not do that full tuning workflow in this accelerated course, but it is important to know why test-set results should be treated carefully.

For today, we will use `max_depth=3` as a reasonable starting point. Later in the notebook, we will compare a few tree depths to understand model complexity, but that comparison is a learning demonstration rather than a full tuning workflow. This tree can ask several questions, but it should still be readable enough to inspect. We will call this model `tree_model`.

## Step 11: Fit the depth-3 decision tree

The code for fitting a decision tree looks a lot like the code you used for logistic regression:

1. create the model object
2. fit the model on the training data
3. use the fitted model to make predictions

The new part is `max_depth=3`, which limits how many layers of decisions the tree can make.

### <font color=0D9488>**Question 4:**</font> Create a `DecisionTreeClassifier` called `tree_model` with `max_depth=3` and `random_state=42`. Then fit it on `X_train_imputed` and `y_train`.

In [ ]:
# Your solution here!

## Step 12: Plot the depth-3 tree

Now we will plot `tree_model`.

This tree will have more boxes than the one-split tree, so read it slowly. Start at the top, choose a branch based on the split rule, and keep moving down until you reach a leaf.

In [ ]:
# Instructor-provided tree plot for the main model
plt.figure(figsize=(18, 9))

plot_tree(
    tree_model,
    feature_names=model_features,
    class_names=["Lower Asthma", "High Asthma"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Depth-3 Decision Tree for High Asthma Classification")
plt.show()

## Reading the depth-3 tree

Use the same reading strategy as before.

Start at the top box:

1. Read the split rule.
2. If the rule is true, move left.
3. If the rule is false, move right.
4. Repeat until you reach a leaf.
5. Read the predicted class in that leaf.

Remember that the tree is using training data to learn these rules. A split can help with prediction without proving that the feature causes asthma burden.

### <font color=0D9488>**Question 5:**</font> Look at the depth-3 tree and choose one path from the top node to a leaf. What sequence of split rules does that path use, and what class does the leaf predict? Then interpret that path in plain language as a rule the model learned. In your interpretation, be careful not to claim that the features cause asthma burden.

**Your solution here!**

**Solution**

One example path is:

1. Start at the root node: `Education <= 9.917`.
2. Follow the `True` branch for rows where `Education <= 9.917`.
3. At the next node, use the split `Poverty <= 18.073`.
4. Follow the `True` branch for rows where `Poverty <= 18.073`.
5. At the next node, use the split `Education <= 5.77`.
6. Follow the `True` branch for rows where `Education <= 5.77`.
7. This path ends at a leaf that predicts `Lower Asthma`.

In plain language, this path says that the model predicts `Lower Asthma` for tracts that follow this sequence of split rules: `Education <= 9.917`, `Poverty <= 18.073`, and `Education <= 5.77`.

The leaf has `value = [1322, 152]`, which means 1,322 lower-asthma rows and 152 high-asthma rows from the training data reached that leaf. Because most training rows in that leaf are lower-asthma rows, the tree predicts `Lower Asthma` for rows that follow this path.

This is a prediction rule learned from this training dataset. It does not show that education or poverty causes asthma burden.

## Step 13: Make predictions with `tree_model`

Now that the tree is fitted, we can use it to predict the class labels for the test set.

We will save the predictions as `tree_preds`.

In [ ]:
# Instructor-provided prediction code
tree_preds = tree_model.predict(X_test_imputed)

print("First 20 predictions:")
print(tree_preds[:20])

## Step 14: Evaluate the depth-3 tree

We can evaluate the decision tree using the same classification metrics from Notebook 5B.

- **Accuracy** tells us the fraction of test rows classified correctly.
- **Precision** focuses on the tracts predicted as high asthma burden. Of those, how many were actually high asthma burden?
- **Recall** focuses on the tracts that were actually high asthma burden. Of those, how many did the model correctly identify?
- **F1** combines precision and recall into one score.

In a public-health setting, recall can be especially important if we are worried about missing tracts that really do have high asthma burden.

### <font color=0D9488>**Question 6:**</font> Calculate accuracy, precision, recall, and F1 for `tree_model` on the test set. Save the results in a dataframe called `tree_metrics`, display it, and plot a confusion matrix.

In [ ]:
# Your solution here!

# Part 4: Compare tree depth, underfitting, and overfitting

A decision tree can be very simple or very flexible depending on how deep it is allowed to grow.

This gives us a useful way to introduce two important modeling ideas about model fit:

- **Underfitting** happens when a model is too simple to capture useful patterns in the data.
- **Overfitting** happens when a model learns details from the training data that do not generalize well to new data.

These ideas apply to many types of machine learning models. In weeks 4 and 5, linear regression and logistic regression gave us useful starting models, but they were limited because they learned relatively simple linear patterns. That simplicity can be helpful for interpretation, but it can also *underfit* if the real pattern is more complex.

Decision trees let us control complexity in a more visible way. A one-split tree can underfit because it only gets one question. A very deep tree can overfit because it can keep making very specific splits, eventually fitting small details in the training data.

We have already fit two trees today:

- `stump_model`, a depth-1 tree that is very simple and easy to read
- `tree_model`, our main depth-3 tree with a few layers of decisions

Now we will fit one more tree:

- `deep_tree_model`, a depth-10 tree that is much more flexible, harder to interpret, and more likely to overfit

The goal is not to find the perfect depth today. The goal is to see how model complexity changes both performance and interpretability.

## Step 15: Fit a deeper tree

The code below fits one more decision tree to the same training data.

This tree uses `max_depth=10`, so it can ask many more questions than the trees we fit earlier. That extra flexibility can help the model fit the training data closely, but it can also make the tree less useful for new data.

In [ ]:
# Instructor-provided deeper tree model
deep_tree_model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Fit the deeper tree on the same training data
deep_tree_model.fit(X_train_imputed, y_train)

## Step 16: Plot the depth-10 tree

Now print the depth-10 tree.

This plot is supposed to feel overwhelming. You are not expected to read every box. Instead, notice how quickly we lose the easy flowchart interpretation that we had with the one-split and depth-3 trees.

As data travels from the top of a deep tree to the leaves, the tree can keep making narrower and narrower groups. That can help it capture detailed patterns, but it can also mean the model is fitting every small nook and cranny of the training data.

### <font color=0D9488>**Question 7:**</font> Plot `deep_tree_model` using `plot_tree()`. Use `model_features` for the feature names and `["Lower Asthma", "High Asthma"]` for the class names. Add a title, then briefly notice how this plot compares to the one-split and depth-3 trees.

In [ ]:
# Your solution here!

Whoa, hard to look at!

This is the interpretability tradeoff in tree models. A small tree is easier for humans to inspect and make decisions from, but it may be too simple. A very deep tree is more flexible, but it can become hard to explain and may not generalize well to test data.

## Step 17: Compare training and test accuracy

Now we will compare how the three trees perform on the training data and the test data.

This comparison helps us think about **model fit**. In machine learning, model fit is about how well a model captures patterns in data. A useful model should learn enough from the training data to make good predictions, but it should also work reasonably well on new data it did not train on.

Training accuracy and test accuracy tell us different things:

- **Training accuracy** tells us how well the model predicts rows it already learned from.
- **Test accuracy** tells us how well the model predicts rows that were held out during training.

A model with low training accuracy and low test accuracy may be **underfitting** because it is too simple to capture useful patterns.

A model with very high training accuracy but noticeably lower test accuracy may be **overfitting** because it learned training-data details that do not generalize well.

For this comparison, we will use accuracy because it gives a simple first look at model fit. In a real public-health setting, we would still care about precision, recall, F1, and the types of errors the model makes.

We will calculate training and test accuracy for each tree depth. The code is provided because the main goal is to interpret the pattern, not to practice writing repeated metric code.

In [ ]:
# Instructor-provided depth comparison code
# Depth-1 tree accuracy
stump_train_acc = accuracy_score(y_train, stump_model.predict(X_train_imputed))
stump_test_acc = accuracy_score(y_test, stump_model.predict(X_test_imputed))

# Depth-3 tree accuracy
tree_train_acc = accuracy_score(y_train, tree_model.predict(X_train_imputed))
tree_test_acc = accuracy_score(y_test, tree_model.predict(X_test_imputed))

# Depth-10 tree accuracy
deep_train_acc = accuracy_score(y_train, deep_tree_model.predict(X_train_imputed))
deep_test_acc = accuracy_score(y_test, deep_tree_model.predict(X_test_imputed))

# Make a comparison table
depth_comparison = pd.DataFrame({
    "Model": ["Depth 1", "Depth 3", "Depth 10"],
    "Training accuracy": [stump_train_acc, tree_train_acc, deep_train_acc],
    "Test accuracy": [stump_test_acc, tree_test_acc, deep_test_acc]
})

display(depth_comparison)

A small plot can make the comparison easier to see.

In [ ]:
# Instructor-provided accuracy comparison plot
depth_comparison.plot(
    x="Model",
    y=["Training accuracy", "Test accuracy"],
    kind="bar",
    figsize=(8, 5)
)

plt.ylabel("Accuracy")
plt.title("Training and Test Accuracy for Different Tree Depths")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

### <font color=0D9488>**Question 8:**</font> This example is a bit subtle, but compare the training accuracy and test accuracy for the three tree depths. For which tree depth do the training and test scores diverge the most? What does that gap suggest about how that tree is fitting the training data compared with how it performs on new test data?

**Your solution here!**

# Part 5: Interpret the model carefully

Decision trees are appealing because we can inspect their decision rules. This interpretability can be useful when models support human decisions.

For example, a public-health team might use a tree-based model as one source of information when identifying tracts that may need more asthma prevention resources, air-quality monitoring, outreach, or deeper investigation. A clinical team might use a similar kind of model to help prioritize which patients need follow-up sooner.

That interpretability does not make the model automatically correct or fair. A tree is still learning associations from the data it was given, and its splits show how the fitted model made predictions, not what caused asthma burden.

This distinction matters because asthma burden is shaped by environmental exposure, housing conditions, access to care, stress, policy choices, racism, segregation, labor conditions, and many other factors. A model can help us describe and predict patterns, but it cannot replace public-health expertise or community knowledge.

In Notebook 6B, you will build on this idea by learning about random forests, which combine many decision trees. You will also return to feature choice more directly and compare models that use different sets of context variables.

# Congratulations, you have completed today's notebook!

## Key Takeaways:

- You rebuilt the `High Asthma` classification target from the numerical `Asthma` column
- You used the same feature set from Week 5 so the main change today was the model type
- You split the data into training and test sets using `stratify=y`
- You imputed missing feature values using the training data median
- You learned that decision trees are nonlinear models that make predictions through yes/no split rules
- You fit a one-split decision tree and practiced reading the printed tree information
- You fit a depth-3 decision tree called `tree_model`
- You plotted decision trees and interpreted split rules, gini, samples, value, and class
- You evaluated `tree_model` using accuracy, a confusion matrix, precision, recall, and F1
- You compared depth-1, depth-3, and depth-10 trees to introduce underfitting and overfitting
- You practiced interpreting a model as a prediction tool rather than a causal explanation

Today’s notebook introduced tree-based classification. Decision trees can be easier to inspect than many models because they show the rules used to make predictions. At the same time, a tree can become too simple or too complex depending on its depth, so model flexibility needs to be evaluated with held-out test data.

## Where This Fits in the ML Workflow

Today’s work fits into the **train model**, **evaluate model**, and **interpret results** parts of the machine learning workflow. You kept the same basic supervised learning setup from Week 5 but replaced logistic regression with a nonlinear decision tree classifier. In Notebook 6B, you will extend this idea by combining many trees into a random forest and thinking more carefully about how feature choices shape model behavior and interpretation.